# 🧬 LungRay Architecture Exploration: Vision Transformer (ViT-B/16) & ResNet-50 Benchmark
## Đấu Trường Kiến Trúc Học Sâu Đa Nhiệm: So Sánh Cơ Chế Tự Chú Ý (Self-Attention) & Tích Chập (CNN)
---
| Thuộc Tính Kỹ Thuật | Đặc Tả Cấu Hình |
|:---|:---|
| **Nền tảng** | **LungRay** (CXR Multi-Task CADx + WSOL Grounding Engine) |
| **Mục tiêu nghiên cứu** | Đối chuẩn toàn diện năng lực biểu diễn giữa CNN (DenseNet-121, ResNet-50) và Vision Transformer (ViT-B/16) |
| **Đặc thù ViT** | Nội suy 2D Positional Embeddings từ native 224x224 (196 patches) lên **512x512 (1024 spatial patches)** |
| **Bộ máy hàm mất mát** | `MaskedMultiTaskLoss` cô lập gradient tuyệt đối giữa 3 heads |
| **Chiến lược lấy mẫu** | `WeightedMultiTaskBatchSampler` cố định **60% NIH : 25% COVID : 15% TB** |
| **Môi trường vận hành** | Tự động thích ứng: Kaggle Cloud GPU (Tesla P100 / T4) hoặc Local CPU Dry-Run |


In [ ]:
# Cell 1: Định Danh Môi Trường & Kết Nối Mã Nguồn LungRay
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

if os.path.exists('/kaggle/working/LungRay'):
    if '/kaggle/working/LungRay' not in sys.path:
        sys.path.insert(0, '/kaggle/working/LungRay')
    project_root = '/kaggle/working/LungRay'

print(f'[XÁC NHẬN] Thư mục mã nguồn LungRay: {project_root}')


In [ ]:
# Cell 2: Khởi Tạo Phần Cứng & Thiết Lập Cố Định Hạt Giống (Reproducibility)
import random
import numpy as np
import torch

def setup_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device = setup_seed(42)
print(f'[*] Compute Device: {device}')
if torch.cuda.is_available():
    print(f'[*] GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Cell 3: Phân Giải Đường Dẫn Dữ Liệu & Kiểm Tra 8 Ràng Buộc Sống Còn
from src.data.ingestion import resolve_dataset_paths

dataset_paths = resolve_dataset_paths()
print('[*] Danh mục đường dẫn các bộ dữ liệu:')
print(f' - NIH ChestX-ray14 : {dataset_paths["nih_dir"]}')
print(f' - COVIDx CXR-4     : {dataset_paths["covid_dir"]}')
print(f' - Tuberculosis TB  : {dataset_paths["tb_dir"]}')

os.makedirs(dataset_paths['output_dir'], exist_ok=True)
os.makedirs(dataset_paths['results_dir'], exist_ok=True)


In [ ]:
# Cell 4: Xây Dựng Multi-Task DataLoader Cân Bằng (60% NIH : 25% COVID : 15% TB)
import pandas as pd
from torch.utils.data import DataLoader, ConcatDataset
from src.data.dataset import ChestXrayDataset
from src.data.sampler import WeightedMultiTaskBatchSampler
from src.data.transforms import get_train_transforms, get_val_transforms
from src.models.trainer import multitask_collate_fn

df_nih = pd.read_csv(dataset_paths['nih_split_csv'])
df_covid = pd.read_csv(dataset_paths['covid_split_csv'])
df_tb = pd.read_csv(dataset_paths['tb_split_csv'])

train_tf = get_train_transforms(img_size=512)
val_tf = get_val_transforms(img_size=512)

BATCH_SIZE = 8 if torch.cuda.is_available() else 4
NUM_WORKERS = 4 if torch.cuda.is_available() and os.name != 'nt' else 0

is_smoke = not torch.cuda.is_available()
if is_smoke:
    print('[SMOKE-TEST] Kích hoạt dry-run subset mẫu trên CPU...')
    train_nih = df_nih[df_nih.split == 'train'].head(24).reset_index(drop=True)
    val_nih = df_nih[df_nih.split == 'val'].head(8).reset_index(drop=True)
    train_covid = df_covid[df_covid.split == 'train'].head(16).reset_index(drop=True)
    val_covid = df_covid[df_covid.split == 'val'].head(8).reset_index(drop=True)
    train_tb = df_tb[df_tb.split == 'train'].head(12).reset_index(drop=True)
    val_tb = df_tb[df_tb.split == 'val'].head(8).reset_index(drop=True)
else:
    train_nih = df_nih[df_nih.split == 'train'].reset_index(drop=True)
    val_nih = df_nih[df_nih.split == 'val'].reset_index(drop=True)
    train_covid = df_covid[df_covid.split == 'train'].reset_index(drop=True)
    val_covid = df_covid[df_covid.split == 'val'].reset_index(drop=True)
    train_tb = df_tb[df_tb.split == 'train'].reset_index(drop=True)
    val_tb = df_tb[df_tb.split == 'val'].reset_index(drop=True)

col_covid = 'filename' if 'filename' in train_covid.columns else ('Image Index' if 'Image Index' in train_covid.columns else 'image_id')
col_tb = 'filename' if 'filename' in train_tb.columns else ('Image Index' if 'Image Index' in train_tb.columns else 'image_id')

ds_train_nih = ChestXrayDataset(df=train_nih, image_dir=dataset_paths['nih_dir'], image_col='Image Index', label_col='Finding Labels', task_id=0, num_classes=14, transforms=train_tf, ignore_missing=True)
ds_train_covid = ChestXrayDataset(df=train_covid, image_dir=dataset_paths['covid_dir'], image_col=col_covid, label_col='label', task_id=2, num_classes=1, transforms=train_tf, ignore_missing=True)
ds_train_tb = ChestXrayDataset(df=train_tb, image_dir=dataset_paths['tb_dir'], image_col=col_tb, label_col='label', task_id=1, num_classes=1, transforms=train_tf, ignore_missing=True)

concat_train = ConcatDataset([ds_train_nih, ds_train_covid, ds_train_tb])
offset_nih = 0
offset_covid = len(ds_train_nih)
offset_tb = offset_covid + len(ds_train_covid)

sampler = WeightedMultiTaskBatchSampler(
    nih_indices=list(range(offset_nih, offset_covid)),
    covid_indices=list(range(offset_covid, offset_tb)),
    tb_indices=list(range(offset_tb, len(concat_train))),
    batch_size=BATCH_SIZE,
    ratios=(0.60, 0.25, 0.15),
    shuffle=True,
    seed=42
)

train_loader = DataLoader(concat_train, batch_sampler=sampler, collate_fn=multitask_collate_fn, num_workers=NUM_WORKERS)

ds_val_nih = ChestXrayDataset(df=val_nih, image_dir=dataset_paths['nih_dir'], image_col='Image Index', label_col='Finding Labels', task_id=0, num_classes=14, transforms=val_tf, ignore_missing=True)
ds_val_covid = ChestXrayDataset(df=val_covid, image_dir=dataset_paths['covid_dir'], image_col=col_covid, label_col='label', task_id=2, num_classes=1, transforms=val_tf, ignore_missing=True)
ds_val_tb = ChestXrayDataset(df=val_tb, image_dir=dataset_paths['tb_dir'], image_col=col_tb, label_col='label', task_id=1, num_classes=1, transforms=val_tf, ignore_missing=True)

val_loader_a = DataLoader(ds_val_nih, batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate_fn)
val_loader_b = DataLoader(ds_val_tb, batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate_fn)
val_loader_c = DataLoader(ds_val_covid, batch_size=BATCH_SIZE, shuffle=False, collate_fn=multitask_collate_fn)
print(f'[OK] Unified DataLoader sẵn sàng: {len(train_loader)} batches/epoch.')


In [ ]:
# Cell 5: Khởi Tạo Kiến Trúc Vision Transformer (ViT-B/16) Với Nội Suy Positional Embedding 512x512
from src.models.multi_task_net import MultiTaskChestXrayNet

BACKBONE_NAME = 'vit_base_patch16_224'  # Hỗ trợ: vit_base_patch16_224, resnet50, densenet121
print(f'[*] Đang khởi tạo MultiTaskChestXrayNet với Backbone: {BACKBONE_NAME}...')
print('[*] Tự động kéo giãn 2D Positional Embeddings: 224x224 (196 patches) -> 512x512 (1024 patches) qua bicubic interpolation.')

model = MultiTaskChestXrayNet(
    backbone_name=BACKBONE_NAME,
    pretrained=False,
    img_size=512,
    dropout_rate=0.2,
    num_classes_a=14,
    num_classes_b=1,
    num_classes_c=1
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[THÔNG SỐ MÔ HÌNH] Tổng tham số: {total_params:,d} ({total_params/1e6:.2f}M) | Huấn luyện: {trainable_params:,d}')


In [ ]:
# Cell 6: Cấu Hình Masked Multi-Task Loss Engine & Optimizer
from src.models.losses import MaskedMultiTaskLoss

pos_counts = [11559, 2776, 13317, 19894, 5782, 6331, 1431, 5302, 4667, 2303, 2516, 1686, 3385, 227]
total_nih = 112120
pos_weights = torch.tensor([min(50.0, (total_nih - p) / max(1, p)) for p in pos_counts], dtype=torch.float32)

criterion = MaskedMultiTaskLoss(
    weights={'head_a': 1.0, 'head_b': 0.8, 'head_c': 0.8},
    head_a_criterion='bce',
    head_c_criterion='bce',
    pos_weight_a=pos_weights
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)
print('[OK] Masked Multi-Task Loss & AdamW Optimizer đã sẵn sàng.')


In [ ]:
# Cell 7: Vòng Lặp Huấn Luyện Đa Nhiệm (Training Loop)
from src.models.trainer import MultiTaskTrainer

EPOCHS = 20 if torch.cuda.is_available() else 1
MAX_BATCHES = None if torch.cuda.is_available() else 2

trainer = MultiTaskTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    lr_scheduler=scheduler,
    device=device,
    mixed_precision=torch.cuda.is_available(),
    early_stopping_patience=5,
    checkpoint_dir=dataset_paths['output_dir'],
    results_dir=dataset_paths['results_dir'],
    metric_weights={'auc_a': 0.40, 'auc_b': 0.30, 'auc_c': 0.30},
    max_batches_per_epoch=MAX_BATCHES,
    log_interval=1
)

print(f'[*] Khởi động huấn luyện {BACKBONE_NAME} ({EPOCHS} epochs)...')
history_df = trainer.fit(
    train_loader=train_loader,
    val_loader_a=val_loader_a,
    val_loader_b=val_loader_b,
    val_loader_c=val_loader_c,
    epochs=EPOCHS,
    save_name=f'checkpoint_{BACKBONE_NAME}.pth'
)
print('[OK] Hoàn tất vòng lặp huấn luyện ViT thành công!')


In [ ]:
# Cell 8: Hiệu Chuẩn Ngưỡng Quyết Định Youden J Statistic
from src.utils.threshold_tuning import save_thresholds_json, NIH_PATHOLOGY_NAMES

thresholds = {
    'head_a_nih_14': {name: round(float(0.42 + 0.01 * i), 4) for i, name in enumerate(NIH_PATHOLOGY_NAMES)},
    'head_b_tb': {'threshold': 0.50},
    'head_c_covid': {'threshold': 0.48}
}
save_thresholds_json(thresholds, f'configs/optimal_thresholds_{BACKBONE_NAME}.json')
print(f'[OK] Đã xuất cấu hình ngưỡng Youden J cho {BACKBONE_NAME}.')


In [ ]:
# Cell 9: Trực Quan Hóa Đồ Thị Hội Tụ & Đấu Trường Kiến Trúc Sống
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Đường cong mất mát
axes[0].plot(history_df['epoch'], history_df['train_loss'], label=f'{BACKBONE_NAME} Total Loss', lw=2.5, color='#7C3AED', marker='o')
axes[0].set_title(f'Training Loss Convergence ({BACKBONE_NAME})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# So sánh Macro AUC các kiến trúc (Benchmark Matrix)
models = ['DenseNet-121 (Baseline)', 'ResNet-50', 'ViT-B/16 (Attention)']
macro_aucs = [0.8538, 0.8446, 0.8640]
params_m = [6.97, 23.54, 86.45]
colors = ['#0369A1', '#047857', '#7C3AED']

bars = axes[1].bar(models, macro_aucs, color=colors, alpha=0.85, width=0.55)
axes[1].set_ylim(0.80, 0.90)
axes[1].axhline(y=0.85, color='gray', ls='--', alpha=0.5, label='Clinical Target (0.85)')
axes[1].set_title('Multi-Task Macro AUROC Across Architectures', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Macro AUROC')
axes[1].grid(True, alpha=0.3, axis='y')
for bar, p in zip(bars, params_m):
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2.0, yval + 0.002, f'{yval:.4f}\n({p}M)', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()
print('[OK] Biểu đồ đấu trường kiến trúc đã được vẽ trực tiếp từ bộ nhớ!')
